# CohortX Task 1 — GEPA Prompt Optimization

Read full-text biomedical articles (PMC `.nxml`) and extract **6 structured fields**:
`conditions`, `study_type`, `sex`, `minimum_age`, `maximum_age`, `eligibility_criteria`
(the last is **50%** of the score).

**What this notebook shows.** We use **GEPA** (DSPy's reflective prompt optimizer) to
automatically rewrite the extraction *instruction* for a small local model
(**Qwen2.5-1.5B via Ollama**), then measure whether it beats the hand-written prompt.

> **Key idea — GEPA is a *dev-time* tool.** A strong model (**Claude**) rewrites the
> prompt *during optimization only*. The shipped system runs **100% offline** on the
> small local model with the single fixed optimized instruction — no API at run time.

Pipeline: **setup → data → baseline → GEPA → optimized eval → before/after → scenario demo.**
GPU-aware (fast on a Colab GPU runtime); also runs locally.

## 0 · Environment & GPU

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print('Running in COLAB' if IN_COLAB else 'Running LOCALLY')
!nvidia-smi -L 2>/dev/null || echo 'No GPU detected — CPU mode (works, just slower)'

## 1 · Get the code
**Colab:** clones the private repo using a GitHub token. Add it under **🔑 Secrets** as
`GITHUB_TOKEN` (a fine-grained PAT with read access to the repo).
**Local:** skipped — you're already in the repo.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    REPO = 'NEILCHENHENRI/cohort-x-task-1'
    if not os.path.isdir('cohort-x-task-1'):
        token = userdata.get('GITHUB_TOKEN')
        !git clone -q https://{token}@github.com/{REPO}.git
    %cd cohort-x-task-1
print('cwd:', os.getcwd())

## 2 · Install dependencies + Ollama

In [ ]:
if IN_COLAB:
    !pip -q install dspy sentence-transformers spacy nltk word2number lxml scikit-learn \
                    pandas openpyxl kagglehub python-dotenv tqdm ollama
    !python -m spacy download en_core_web_sm -q
    !curl -fsSL https://ollama.com/install.sh | sh
# NLTK corpora for the FM3S eligibility metric (certifi avoids macOS/Colab SSL errors)
import certifi, nltk
os.environ['SSL_CERT_FILE'] = certifi.where()
for pkg in ('wordnet', 'wordnet_ic', 'omw-1.4'):
    nltk.download(pkg, quiet=True)
print('dependencies ready')

## 3 · Start Ollama & pull Qwen2.5-1.5B

In [ ]:
import subprocess, time, requests
if IN_COLAB:
    subprocess.Popen(['ollama', 'serve'])
    time.sleep(5)
    !ollama pull qwen2.5:1.5b
print('Ollama server up:', requests.get('http://localhost:11434/api/tags').ok)

## 4 · Secrets (Kaggle + Anthropic)
Add under **🔑 Secrets**: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `ANTHROPIC_API_KEY`.
The Kaggle account must have **joined** the competition (accepted its rules).

In [ ]:
if IN_COLAB:
    import json
    from google.colab import userdata
    os.makedirs('/root/.kaggle', exist_ok=True)
    json.dump({'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')},
              open('/root/.kaggle/kaggle.json', 'w'))
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    open('.env', 'w').write(f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n")
    print('secrets configured')
else:
    print('Local run: using ~/.kaggle/kaggle.json and .env already on disk')

## 5 · Download the competition data

In [ ]:
!python download_data.py

## 6 · Baseline — the current hand-written prompt
Runs the **shipped offline path** (`predict_ollama.py`) on 60 held-out papers and scores
it. This is the **"before"** number GEPA must beat.

In [ ]:
!python run_eval.py --split holdout --output results/baseline.json
import json
baseline = json.load(open('results/baseline.json'))['mean_scores']
print('Baseline overall:', baseline['overall'])
baseline

## 7 · Run GEPA (dev-time optimization)
Claude reflects on the local model's mistakes (using our **field-specific feedback**) and
rewrites the instruction. Cheap on `auto`/light budgets — the run prints the cost.

In [ ]:
!python optimize_gepa.py --max_metric_calls 150 --out results/gepa_optimized.json
print('\n===== OPTIMIZED INSTRUCTION =====\n')
print(open('results/optimized_instruction.txt').read())

## 8 · Evaluate the optimized instruction (offline, shipped path)
We swap the optimized instruction into the **same offline path** and score the **same**
held-out papers — an honest, apples-to-apples comparison.

In [ ]:
!python run_eval.py --split holdout --instruction_file results/optimized_instruction.txt \
                    --output results/optimized.json

## 9 · Before / After

In [ ]:
import json, pandas as pd
b = json.load(open('results/baseline.json'))['mean_scores']
o = json.load(open('results/optimized.json'))['mean_scores']
df = pd.DataFrame({'baseline': b, 'optimized': o})
df['delta'] = (df['optimized'] - df['baseline'])
df = df.reindex(['eligibility_criteria','conditions','study_type',
                 'minimum_age','maximum_age','sex','overall'])
df.round(4)

## 10 · Scenario demo — extract from one real paper
Runs the **optimized** system on a held-out article and shows the structured output.

In [ ]:
import importlib, predict_ollama, json
importlib.reload(predict_ollama)
from common.parser import NXMLParser
from gepa_opt.data_split import get_splits, nxml_path

prose = open('results/optimized_instruction.txt').read().strip()
predict_ollama.INSTRUCTION_PROSE = prose
predict_ollama.INSTRUCTION = f"{prose}\n\n{predict_ollama.OUTPUT_SCHEMA}"

pid = get_splits()['holdout'][0]
parsed = NXMLParser().parse(str(nxml_path(pid)))
pred = predict_ollama.extract(pid, parsed, 'qwen2.5:1.5b')
print('PMC' + pid)
print(json.dumps(pred, indent=2)[:1800])